# Exploring Delta Live Tables (Now Lakeflow Pipeline)
## Exploring Delta Live Tables
Declarative ETL framework powered by Apache Spark for building reliable and maintainable data pipelines.<br>
Benefits include:
- Simplified pipeline construction
- Maintain table dependencies
- Support for data quality control

Some differences between Spark Structured Streaming and DLT, no sql support and basic data quality control use delta lake table constraints

## DLT Object types
- Streaming tables-- new data
```sql
CREATE OR REFRESH STREAMING TABLE my_table AS
SELECT * FROM <streaming source, autoloader stream/append-only delta>

CREATE OR REFRESH STREAMING TABLE table_2 AS
SELECT * FROM STREAM(table_1)
```
- Materailized views-- reprocess the entire source dataset everytime
```sql
CREATE OR REPLACE MATERIALIZED VIEW my_mview AS
SELECT * FROM <batch source>
```
- Live views-- temp view objects scoped to DLT pipeline (ideal for intermediate data transformations and quality checks)
```sql
CREATE TEMPORARY LIVE VIEW my_temp_view AS 
<query>
```

## DLT Expectations
Constraints- enforcing data quality CONSTRAINT

```sql
CONSTRAINT <constraint_name> EXPECT (<condition>) ON VIOLATION <action>
```
ON VIOLATION
- DROP ROW-- discards records that do not meet the specified constraint
- FAIL UPDATE-- any violation causes the entire pipeline to fail (data quality is critical, any deviation must be addressed immediately)
- *No action {default)*-- reported in metrics, but included in dataset

## Implementing DLT
Let's get hands on and start delving into Delta Live Tables<br>
DLT is now known as Lakeflow ETL Pipelines
DLT pipelines are implemented using notebooks written in Python or SQL


In [0]:
SET school.dataset_path = 'abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/mnt/DE-Associate-Book/datasets/school'

### Create Streaming Table

In [0]:
CREATE OR REFRESH STREAMING TABLE enrollments_raw
COMMENT "The raw courses enrollments, ingested from enrollments-dlt-raw folder"
AS SELECT * FROM cloud_files("${school.dataset_path}/enrollments-dlt-raw", "json", map("cloudFiles.inferColumnTypes", "true"))


### Creating a materialized view
Let's create a lookup table for the silver tier

In [0]:
CREATE OR REPLACE MATERIALIZED VIEW students_mv 
COMMENT "The students lookup table, ingested from students-json"
AS SELECT * FROM read_files("${school.dataset_path}/students-json", format => "json", inferColumnTypes => true)

Silver Layer

In [0]:
CREATE OR REFRESH STREAMING TABLE enrollments_cleaned (
  CONSTRAINT valid_order_number EXPECT (enroll_id IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "The cleaned courses enrollments with valid enroll_id"
AS 
  SELECT enroll_id, quantity, o.student_id,
          c.profile:first_name as f_name, c.profile:last_name as l_name,
          cast(from_unixtime(enroll_timestamp,
            'yyyy-MM-dd HH:mm:ss') AS timestamp) AS formatted_timestamp,
          o.courses, c.profile:address:country AS country
  FROM STREAM(LIVE.enrollments_raw) o
  LEFT JOIN LIVE.students_mv c
    ON o.student_id = c.student_id

In [0]:
CREATE OR REPLACE MATERIALIZED VIEW uk_daily_student_courses
COMMENT "Daily number of courses per student in United Kingdom"
AS
  SELECT student_id, f_name, l_name,
          date_trunc("DD", formatted_timestamp) order_date,
          sum(quantity) courses_counts
  FROM LIVE.enrollments_cleaned
  WHERE country = "United Kingdom"
  GROUP BY student_id, f_name, l_name, date_trunc("DD", formatted_timestamp)

In [0]:
CREATE OR REPLACE MATERIALIZED VIEW fr_daily_student_courses
COMMENT "Daily number of courses per student in France"
AS
 SELECT student_id, f_name, l_name,
        date_trunc("DD", formatted_timestamp) order_date,
        sum(quantity) courses_counts
 FROM LIVE.enrollments_cleaned
 WHERE country = "France"
 GROUP BY student_id, f_name, l_name, date_trunc("DD", formatted_timestamp)